In [1]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("../data")

game_file = next(DATA_DIR.glob("game_data_public.*.csv.gz"))
replay_file = next(DATA_DIR.glob("replay_data_public.*.csv.gz"))

print("Game file:", game_file.name)
print("Replay file:", replay_file.name)

# Load a manageable sample first
GAME_KEY = ["draft_id", "match_number", "game_number"]

game_df = pd.read_csv(
    game_file,
    nrows=100,
)

wanted_keys = set(
    map(
        tuple,
        game_df[GAME_KEY].itertuples(index=False, name=None)
    )
)

replay_parts = []

for chunk in pd.read_csv(
    replay_file,
    chunksize=10_000,
    low_memory=False,
):
    keys = list(
        zip(
            chunk["draft_id"],
            chunk["match_number"],
            chunk["game_number"],
        )
    )

    mask = [key in wanted_keys for key in keys]

    if any(mask):
        replay_parts.append(chunk.loc[mask])

replay_df = pd.concat(
    replay_parts,
    ignore_index=True,
)

print("game rows:", len(game_df))
print("replay rows:", len(replay_df))

Game file: game_data_public.Cube_-_Powered.PremierDraft.csv.gz
Replay file: replay_data_public.Cube_-_Powered.PremierDraft.csv.gz
game rows: 100
replay rows: 100
